In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

from transforms.silver import parse_swedish_decimal, extract_match_row

BRONZE_PATH = "/Volumes/stryktipset/bronze/raw_draws/draw_*.json"
SILVER_TABLE = "stryktipset.silver.matches"


In [0]:
def main():
    draws = spark.read.option("multiLine", True).json(BRONZE_PATH).filter(F.col("error").isNull())
    matches = draws.withColumn("event", F.explode("draw.drawEvents"))

    result_df = matches.select(extract_match_row(F.col("draw.drawNumber"), F.col("event")))
    result_df = result_df.filter(F.col("home_goals").isNotNull() & F.col("away_goals").isNotNull())

    latest_per_match = Window.partitionBy('match_id').orderBy(F.col('draw_number').desc())
    result_df = (
        result_df
        .withColumn('row_num', F.row_number().over(latest_per_match))
        .filter(F.col('row_num') == 1)
        .drop('row_num')
    )

    result_df.write.format("delta").mode("overwrite").saveAsTable(SILVER_TABLE)

main()
